# MIMIC-IV cohort extraction

Original cohort construction from the independent UCLA project, with row previews removed. Run only with authorized MIMIC-IV BigQuery access. See README.md for the retrospective feature-time limitations. This notebook does not enforce an admission-time prediction window. Outputs must remain private.


In [ ]:
from google.cloud import bigquery
import pandas as pd

In [ ]:
# Set client
client = bigquery.Client()

In [ ]:
# 1.icustays table
query_icustays = f"""
SELECT *
FROM `physionet-data.mimiciv_3_1_icu.icustays`
ORDER BY subject_id, hadm_id, stay_id
"""

icustays_tble = client.query(query_icustays).to_dataframe()

In [ ]:
# 2.admissions table
query_admissions = f"""
SELECT *
FROM `physionet-data.mimiciv_3_1_hosp.admissions`
ORDER BY subject_id, hadm_id
"""
admissions_tble = client.query(query_admissions).to_dataframe()

In [ ]:
# 3.patients table
query_patients = f"""
SELECT *
FROM `physionet-data.mimiciv_3_1_hosp.patients`
ORDER BY subject_id
"""
patients_tble = client.query(query_patients).to_dataframe()

In [ ]:
# 4.labevents table
lab_items = [50912, 50971, 50983, 50902, 50882, 51221, 51301, 50931]

query_labevents = f"""
WITH eligible_labs AS (
  SELECT
    le.subject_id,
    icu.stay_id,
    le.itemid,
    le.valuenum,
    le.storetime,
    icu.intime,
    ROW_NUMBER() OVER (
      PARTITION BY le.subject_id, icu.stay_id, le.itemid
      ORDER BY le.storetime DESC
    ) AS rn
  FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
  INNER JOIN (
    SELECT subject_id, stay_id, intime
    FROM `physionet-data.mimiciv_3_1_icu.icustays`
  ) icu
    ON le.subject_id = icu.subject_id
  WHERE le.itemid IN ({','.join(str(i) for i in lab_items)})
    AND le.storetime < icu.intime
)
SELECT
  subject_id,
  stay_id,
  MAX(CASE WHEN itemid = 50912 THEN valuenum END) AS lab_50912,
  MAX(CASE WHEN itemid = 50971 THEN valuenum END) AS lab_50971,
  MAX(CASE WHEN itemid = 50983 THEN valuenum END) AS lab_50983,
  MAX(CASE WHEN itemid = 50902 THEN valuenum END) AS lab_50902,
  MAX(CASE WHEN itemid = 50882 THEN valuenum END) AS lab_50882,
  MAX(CASE WHEN itemid = 51221 THEN valuenum END) AS lab_51221,
  MAX(CASE WHEN itemid = 51301 THEN valuenum END) AS lab_51301,
  MAX(CASE WHEN itemid = 50931 THEN valuenum END) AS lab_50931
FROM eligible_labs
WHERE rn = 1
GROUP BY subject_id, stay_id
"""

labevents_cohort = client.query(query_labevents).to_dataframe()

In [ ]:
# 5.chartevents table
chart_items = [220045, 220179, 220180, 223761, 220210]

query_chartevents = f"""
WITH chart_first AS (
  SELECT
    ce.subject_id,
    icu.stay_id,
    ce.itemid,
    ce.valuenum,
    ce.storetime,
    icu.intime,
    icu.outtime,
    ROW_NUMBER() OVER (
      PARTITION BY icu.stay_id, ce.itemid
      ORDER BY ce.storetime ASC
    ) AS rn,
    MIN(ce.storetime) OVER (
      PARTITION BY icu.stay_id, ce.itemid
    ) AS first_storetime
  FROM `physionet-data.mimiciv_3_1_icu.chartevents` ce
  INNER JOIN (
    SELECT subject_id, stay_id, intime, outtime
    FROM `physionet-data.mimiciv_3_1_icu.icustays`
  ) icu
    ON ce.stay_id = icu.stay_id
  WHERE ce.itemid IN ({','.join(str(i) for i in chart_items)})
    AND ce.storetime >= icu.intime
    AND ce.storetime <= icu.outtime
)
, chart_agg AS (
  SELECT
    subject_id,
    stay_id,
    itemid,
    AVG(valuenum) AS valuenum
  FROM chart_first
  WHERE storetime = first_storetime
  GROUP BY subject_id, stay_id, itemid
)
SELECT
  subject_id,
  stay_id,
  MAX(CASE WHEN itemid = 220045 THEN valuenum END) AS chart_220045,
  MAX(CASE WHEN itemid = 220179 THEN valuenum END) AS chart_220179,
  MAX(CASE WHEN itemid = 220180 THEN valuenum END) AS chart_220180,
  MAX(CASE WHEN itemid = 223761 THEN valuenum END) AS chart_223761,
  MAX(CASE WHEN itemid = 220210 THEN valuenum END) AS chart_220210
FROM chart_agg
GROUP BY subject_id, stay_id
"""

chartevents_cohort = client.query(query_chartevents).to_dataframe()

Put things together

In [ ]:
# 1. Start with icustays_tble, merge in admissions
mimic_icu_cohort = icustays_tble.merge(
    admissions_tble,
    on=["subject_id", "hadm_id"],
    how="inner"
)

# 2. Merge in patients
mimic_icu_cohort = mimic_icu_cohort.merge(
    patients_tble,
    on="subject_id",
    how="inner"
)

# 3. Keep only adults (age >= 18)
mimic_icu_cohort = mimic_icu_cohort[mimic_icu_cohort['anchor_age'] >= 18]

# 4. Merge in labevents
mimic_icu_cohort = mimic_icu_cohort.merge(
    labevents_cohort,
    on=["subject_id", "stay_id"],
    how="left"
)

# 5. Merge in chartevents
mimic_icu_cohort = mimic_icu_cohort.merge(
    chartevents_cohort,
    on=["subject_id", "stay_id"],
    how="left"
)

# 6. Sort by subject_id, hadm_id, stay_id
mimic_icu_cohort = mimic_icu_cohort.sort_values(
    by=["subject_id", "hadm_id", "stay_id"]
)

Preprocessing

In [ ]:
lab_chart_mapping = {
    "lab_50902": "chloride",
    "lab_50912": "creatinine",
    "lab_50983": "sodium",
    "lab_50971": "potassium",
    "lab_50931": "glucose",
    "lab_51221": "hematocrit",
    "lab_51301": "wbc",
    "lab_50882": "bicarbonate",
    "chart_220179": "non_invasive_blood_pressure_systolic",
    "chart_220180": "non_invasive_blood_pressure_diastolic",
    "chart_220210": "respiratory_rate",
    "chart_223761": "temperature_fahrenheit",
    "chart_220045": "heart_rate"
}
mimic_icu_cohort= mimic_icu_cohort.rename(columns=lab_chart_mapping)

In [ ]:
def lump_top(series, n=4, other_label="Other"):
    top = series.value_counts().nlargest(n).index
    return series.where(series.isin(top), other_label)

In [ ]:
mimic_icu_cohort["first_careunit"] = lump_top(mimic_icu_cohort["first_careunit"], 4)
mimic_icu_cohort["last_careunit"] = lump_top(mimic_icu_cohort["last_careunit"], 4)
mimic_icu_cohort["admission_type"] = lump_top(mimic_icu_cohort["admission_type"], 4)
mimic_icu_cohort["admission_location"] = lump_top(mimic_icu_cohort["admission_location"], 3)
mimic_icu_cohort["discharge_location"] = lump_top(mimic_icu_cohort["discharge_location"], 4)
mimic_icu_cohort["insurance"] = lump_top(mimic_icu_cohort["insurance"], 4)
mimic_icu_cohort["language"] = lump_top(mimic_icu_cohort["language"], 24)
mimic_icu_cohort["marital_status"] = lump_top(mimic_icu_cohort["marital_status"], 4, "Unknown")

In [ ]:
def collapse_race(race):
    asian = [
        "ASIAN", "ASIAN - ASIAN INDIAN", "ASIAN - CHINESE", "ASIAN - KOREAN", "ASIAN - SOUTH EAST ASIAN"
    ]
    black = [
        "BLACK", "BLACK/AFRICAN AMERICAN", "BLACK/AFRICAN", "BLACK/CAPE VERDEAN", "BLACK/CARIBBEAN ISLAND"
    ]
    hispanic = [
        "HISPANIC OR LATINO", "HISPANIC/LATINO - CENTRAL AMERICAN", "HISPANIC/LATINO - COLUMBIAN",
        "HISPANIC/LATINO - CUBAN", "HISPANIC/LATINO - DOMINICAN", "HISPANIC/LATINO - GUATEMALAN",
        "HISPANIC/LATINO - HONDURAN", "HISPANIC/LATINO - MEXICAN", "HISPANIC/LATINO - PUERTO RICAN",
        "HISPANIC/LATINO - SALVADORAN"
    ]
    white = [
        "WHITE", "WHITE - BRAZILIAN", "WHITE - EASTERN EUROPEAN", "WHITE - OTHER EUROPEAN", "WHITE - RUSSIAN"
    ]
    if race in asian:
        return "ASIAN"
    elif race in black:
        return "BLACK"
    elif race in hispanic:
        return "HISPANIC"
    elif race in white:
        return "WHITE"
    else:
        return "Other"

mimic_icu_cohort["race"] = mimic_icu_cohort["race"].map(collapse_race)

In [ ]:
mimic_icu_cohort = mimic_icu_cohort[mimic_icu_cohort["los"].notna()]
mimic_icu_cohort["los_long"] = mimic_icu_cohort["los"] >= 2

In [ ]:
from pathlib import Path
Path("data").mkdir(exist_ok=True)
mimic_icu_cohort.to_parquet("data/mimiciv_icu_cohort.parquet", index=False)